# Notebook Chat (via Reclaim Cloud OpenAI Proxy)

This notebook provides a small **chat UI inside Jupyter** using `ipywidgets`, but it sends requests to **your Reclaim Cloud proxy** instead of calling OpenAI directly.

**Why:** Binder notebooks are built from public GitHub repos, so you should not store an OpenAI API key in the repo. Your proxy keeps the key server-side.

---

## What you need
- Your proxy base URL (example):
  - `http://node17350-gettysburg-openai-proxy.us.reclaim.cloud`
- A model name that your proxy allows (e.g. `gpt-5.2`)
- *(Optional)* a workshop code, if you enabled that on the proxy.

---

## Quick test
Run the next cell, then run the **Smoke test** cell. If it works, proceed to the **Chat UI** section.


In [ ]:
import os
import getpass
import requests

# --- Set your proxy URL here ---
PROXY_URL = os.getenv("PROXY_URL", "http://node17350-gettysburg-openai-proxy.us.reclaim.cloud").rstrip("/")

# --- Default model (must be in ALLOWED_MODELS on your proxy) ---
DEFAULT_MODEL = os.getenv("MODEL", "gpt-5.2")

# --- Optional workshop code (if your proxy enforces WORKSHOP_CODE) ---
# You can set WORKSHOP_CODE as an env var, or you'll be prompted the first time you run the chat.
WORKSHOP_CODE = os.getenv("WORKSHOP_CODE", "").strip() or None

def proxy_chat(messages, model=DEFAULT_MODEL, workshop_code=WORKSHOP_CODE, timeout=120):
    """Send messages to the proxy. Returns assistant text."""
    payload = {
        "messages": messages,
        "model": model,
        "workshop_code": workshop_code,
    }
    r = requests.post(f"{PROXY_URL}/chat", json=payload, timeout=timeout)
    # Helpful error message on failure
    try:
        r.raise_for_status()
    except Exception as e:
        raise RuntimeError(f"Proxy call failed: {e}\nStatus: {r.status_code}\nBody: {r.text[:500]}") from e
    data = r.json()
    return data.get("text", "")


## Smoke test

If this returns a short response, your proxy is wired correctly.

In [ ]:
proxy_chat([{"role": "user", "content": "Say hello in one sentence."}], model=DEFAULT_MODEL)

## If you enabled a workshop code on the proxy

If your proxy requires a code, run this cell once per session to set it (it won’t echo).  
If you didn't enable one, you can skip this.

In [ ]:
if WORKSHOP_CODE is None:
    code = getpass.getpass("Workshop code (leave blank if none): ").strip()
    WORKSHOP_CODE = code or None
WORKSHOP_CODE

## Chat UI (Option A)

This creates a simple chat panel inside the notebook output.  
- **Send** posts your message to the proxy
- **Clear** resets the conversation (keeps the system prompt)

Tip: If you want to switch models, use the dropdown before sending.


In [ ]:
import ipywidgets as widgets
from IPython.display import display

# You can edit this list to match your proxy allowlist
MODEL_OPTIONS = [
    DEFAULT_MODEL,
    "gpt-5.2-pro",
    "gpt-5.2",
    "gpt-5",
    "gpt-4.1",
    "gpt-4.1-mini",
]

# Avoid duplicates while preserving order
seen = set()
MODEL_OPTIONS = [m for m in MODEL_OPTIONS if not (m in seen or seen.add(m))]

system_prompt = "You are a helpful assistant embedded in a Jupyter notebook."

history = [{"role": "system", "content": system_prompt}]

title = widgets.HTML("<b>Notebook Chat</b>")
chat_box = widgets.Output(layout={"border": "1px solid #ddd", "padding": "10px", "height": "340px", "overflow_y": "auto"})
user_input = widgets.Textarea(placeholder="Type a message…", layout=widgets.Layout(width="100%", height="90px"))
send_btn = widgets.Button(description="Send", button_style="primary")
clear_btn = widgets.Button(description="Clear")
model_dd = widgets.Dropdown(options=MODEL_OPTIONS, value=DEFAULT_MODEL, description="Model:")
status = widgets.HTML("")

def render(role, content):
    who = "You" if role == "user" else ("Assistant" if role == "assistant" else "System")
    return widgets.HTML(f"""<div style="margin:8px 0;">
        <div><b>{who}:</b></div>
        <div style="white-space: pre-wrap;">{content}</div>
    </div>""")

def on_send(_=None):
    global history, WORKSHOP_CODE
    text = user_input.value.strip()
    if not text:
        return
    user_input.value = ""
    status.value = "<i>Thinking…</i>"

    # If a workshop code is required and not set, prompt once.
    if WORKSHOP_CODE is None:
        import getpass
        code = getpass.getpass("Workshop code (leave blank if none): ").strip()
        WORKSHOP_CODE = code or None

    history.append({"role": "user", "content": text})
    with chat_box:
        display(render("user", text))

    try:
        reply = proxy_chat(history, model=model_dd.value, workshop_code=WORKSHOP_CODE)
    except Exception as e:
        reply = str(e)

    history.append({"role": "assistant", "content": reply})
    with chat_box:
        display(render("assistant", reply))

    status.value = ""

def on_clear(_=None):
    global history
    history = [{"role": "system", "content": system_prompt}]
    chat_box.clear_output()
    status.value = ""

send_btn.on_click(on_send)
clear_btn.on_click(on_clear)

controls = widgets.HBox([send_btn, clear_btn])
ui = widgets.VBox([title, model_dd, chat_box, user_input, controls, status])
display(ui)


## Notes

- If you get **401 Unauthorized**, your proxy likely expects a workshop code (or it didn’t match).
- If you get **400 Model not allowed**, edit `MODEL_OPTIONS` to match your proxy `ALLOWED_MODELS`, or change `DEFAULT_MODEL`.
- If you see widget display issues in Binder, make sure your Binder environment includes `ipywidgets`.
